# 3. The Parallel Chain (RunnableParallel)

A **parallel chain** runs several branches **at the same time** on the **same input**, then collects
their results into a dict. Use it when tasks are independent — no reason to wait for one before
starting another.

---

## 1. Simple Definition

> **Kid version:** Imagine giving the same photo to three friends at once: one counts the people, one
> describes the colors, one guesses the location. They all work **at the same time** and you collect
> their three answers together. That's a parallel chain.

**Professional definition:** A `RunnableParallel` (also written as a plain dict in LCEL) runs multiple
Runnables **concurrently on the same input** and returns a dictionary mapping each key to its branch's
output.

```python
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    summary=summarize_chain,
    keywords=keyword_chain,
    sentiment=sentiment_chain,
)
parallel.invoke({"text": article})
# → {"summary": "...", "keywords": [...], "sentiment": "positive"}
```

---

## 2. Why Does It Exist?

**The problem:** When you have several **independent** operations on the same input, doing them one at
a time is slow — you pay the latency of each in series for no reason.

### Before (sequential — slow, and awkward to merge)

```python
summary   = summarize_chain.invoke({"text": article})   # wait...
keywords  = keyword_chain.invoke({"text": article})     # then wait...
sentiment = sentiment_chain.invoke({"text": article})   # then wait...
result = {"summary": summary, "keywords": keywords, "sentiment": sentiment}
# total time ≈ sum of all three
```

### After (parallel — concurrent, auto-merged)

```python
parallel = RunnableParallel(summary=summarize_chain,
                            keywords=keyword_chain,
                            sentiment=sentiment_chain)
result = parallel.invoke({"text": article})
# total time ≈ the SLOWEST branch (they overlap), result already a dict
```

Independent branches overlap, so wall-clock time drops from *sum* to roughly the *max* of the
branches — and you get a tidy dict back.

**Where you'll use it:** multi-aspect analysis (summary + sentiment + keywords), generating several
variants at once, **RAG** (retrieve context **and** pass the question through together — the classic
pattern), fanning one input to many models.

---

## 3. Real-Life Analogy

**A newspaper editor assigning one event to several reporters** 📰. The sports writer, the photographer,
and the fact-checker all work on the same event **simultaneously**, then the editor collects their
pieces into one story. Nobody waits in line — that's the speed win of parallel.

Other analogies: a **pit crew** (four people change four tires at once), an **exam** graded by multiple
markers in parallel.

---

## 4. Where It Fits in LangChain Architecture

```
                 ┌───────────► summarize_chain ──► "summary"
   same input ──►┤
   {"text":...}  ├───────────► keyword_chain   ──► "keywords"
                 └───────────► sentiment_chain  ──► "sentiment"
                                     │
                                     ▼
                    {"summary":..., "keywords":..., "sentiment":...}   (a dict)
```

`RunnableParallel` is the **fan-out/gather** engine. It's often used *inside* a sequential chain (fan
out, then a later step consumes the merged dict).

---

## 5. Internal Working

```
  parallel.invoke({"text": article})
        │
        ▼
  the SAME input dict is handed to EVERY branch
        │
        ├── branch "summary"   runs ─┐
        ├── branch "keywords"  runs ─┤  (concurrently — threads/async under the hood)
        └── branch "sentiment" runs ─┘
        │
        ▼
  wait for all branches to finish
        │
        ▼
  assemble outputs by key → {"summary":..., "keywords":..., "sentiment":...}
```

Two important facts:
- **Every branch receives the exact same input** (not each other's output — that's what makes them
  independent and parallelizable).
- The **output keys** are the dict keys you chose; each value is that branch's result.

---

## 6. The dict shorthand (you'll see this everywhere)

In LCEL, a **plain dict of Runnables is automatically a `RunnableParallel`**. These two are identical:

```python
from langchain_core.runnables import RunnableParallel

# Explicit:
RunnableParallel(summary=summarize, keywords=keywords)

# Shorthand (a dict) — LangChain coerces it to RunnableParallel:
{"summary": summarize, "keywords": keywords}
```

This is why you often see a `{...}` dict sitting in the middle of a chain — it's a parallel step.

---

## 7. Key parts / patterns

### Named branches

**Definition:** Each `key=runnable` pair is a branch; the key names its output.

**Why it exists:** So results come back clearly labeled in a dict.

**When developers use it:** Any time independent outputs are needed together.

```python
RunnableParallel(joke=joke_chain, fact=fact_chain)   # → {"joke": ..., "fact": ...}
```

---

### Combining with RunnablePassthrough (keep the input too)

**Definition:** Include the original input alongside computed branches.

**Why it exists:** A later step often needs both the raw input and the new results.

**When developers use it:** RAG and multi-stage pipelines.

```python
from langchain_core.runnables import RunnablePassthrough

RunnableParallel(
    context=retriever,               # fetch docs
    question=RunnablePassthrough(),  # keep the original question unchanged
)
# → {"context": [...docs...], "question": "the original question"}
```

This exact pattern is the backbone of RAG chains.

---

### Feeding the merged dict into a next step (parallel inside sequential)

**Definition:** A parallel step's dict output becomes the input to a following prompt/step.

**Why it exists:** Gather several inputs, then reason over all of them together.

```python
prompt = ChatPromptTemplate.from_template(
    "Question: {question}\n\nUse this context:\n{context}\n\nAnswer:"
)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}   # parallel fan-out
    | prompt | model | StrOutputParser()                        # then sequential
)
rag_chain.invoke("What is LCEL?")
```

---

## 8. Worked example — multi-aspect analysis

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

def make(instruction):
    return ChatPromptTemplate.from_template(instruction + "\n\n{text}") | model | StrOutputParser()

analyze = RunnableParallel(
    summary   = make("Summarize in one sentence:"),
    sentiment = make("One word sentiment (positive/negative/neutral):"),
    title     = make("Suggest a catchy title for:"),
)

result = analyze.invoke({"text": "LangChain makes it easy to compose LLM pipelines..."})
print(result["summary"], "|", result["sentiment"], "|", result["title"])
# All three produced concurrently, returned as one dict.
```

---

## 9. When NOT to use parallel

- When a branch **needs another branch's output** → that's a **dependency** → use a **sequential**
  chain. Parallel branches can't see each other.
- When branches share a strict rate limit and concurrency would trigger throttling — then serialize or
  cap concurrency.

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [3]:
# step1: create the prompt templates
short_notes_prompt = PromptTemplate(template='Generate short and simple notes from the following text \n {text}',
                                    input_variables=['text'])

quiz_prompt  = PromptTemplate(template='Generate 5 short question answers from the following text \n {text}',
                              input_variables=['text'])

merge_prompt = PromptTemplate(template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
                              input_variables=['notes', 'quiz'])

In [4]:
# step2: Initialize the language model
llm = ChatOllama(model="qwen3:8b")

In [5]:
# step3: Initialize output parser
output_parser = StrOutputParser()

In [6]:
# step4: Create the chain
parallel_chain = RunnableParallel({
                                'notes': short_notes_prompt | llm | output_parser,
                                'quiz': quiz_prompt | llm | output_parser
})

merge_chain = merge_prompt | llm | output_parser

chain = parallel_chain | merge_chain

In [7]:
text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})

In [8]:
print(result)

**Support Vector Machines (SVMs): Notes & Quiz**  

---

### **Overview**  
Support Vector Machines (SVMs) are **supervised learning methods** used for **classification, regression, and outlier detection**. They are particularly effective in scenarios where data is high-dimensional or sparse.  

---

### **Advantages**  
1. **High-Dimensional Effectiveness**: SVMs excel in **high-dimensional spaces** (e.g., text classification, image recognition).  
2. **Memory Efficiency**: They use only a subset of training points called **support vectors**, making them memory efficient.  
3. **Versatility**: Support custom kernels (e.g., linear, RBF, polynomial) to handle non-linear relationships.  
4. **Robustness to Overfitting**: When dimensions > samples, SVMs remain effective by focusing on support vectors.  

---

### **Disadvantages**  
1. **Overfitting Risk**: If features exceed samples, overfitting can occur. Requires careful kernel selection and regularization.  
2. **No Direct Probability

In [9]:
chain.get_graph().print_ascii()

            +---------------------------+            
            | Parallel<notes,quiz>Input |            
            +---------------------------+            
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                    +------------+   
  | ChatOllama |                    | ChatOllama |   
  +------------+                    +------------+   
          *                               *          
          *                               *          
          *                               *          
+-----------------+         

In [10]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b")

output_parser = StrOutputParser()

summary_chain = (ChatPromptTemplate.from_template("Summarize this in one sentence:\n{text}")| llm | output_parser)

sentiment_chain = (ChatPromptTemplate.from_template("Determine the sentiment of this text:\n{text}")| llm | output_parser)

parallel_chain = RunnableParallel(summary=summary_chain, sentiment=sentiment_chain)

result = parallel_chain.invoke({"text": "I really enjoyed this product. It was fast and easy to use."})

print(result)

{'summary': 'The product was enjoyable, fast, and easy to use.', 'sentiment': 'The sentiment of the text is **positive**.  \n\nThe phrases "I really enjoyed this product" and "It was fast and easy to use" clearly express satisfaction and approval, with words like "enjoyed," "fast," and "easy" conveying a favorable tone. There are no negative or neutral indicators in the text.'}
